# SASRec, 2019

In [2]:
import os, sys

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf

sys.path.insert(0, '..')
from evaluate import ndcg, hr

2023-09-07 17:22:52.282468: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-09-07 17:22:53.127468: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


# Data Load 

In [3]:
movielens_dir = '../data/ml-1m/'
max_length = 200

In [4]:
df = pd.read_csv(os.path.join(movielens_dir, 'ratings.dat'), sep='::', header=None)
df.columns = ['user_id', 'item_id', 'rating', 'timestamp']
print(df.shape)
df.head()

/tmp/ipykernel_3174/1796806906.py:1: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv(os.path.join(movielens_dir, 'ratings.dat'), sep='::', header=None)


(1000209, 4)


,user_id,item_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


전체 데이터 수 확인, pos, neg 데이터 분배

In [5]:
all_users = list(set(df['user_id']))
all_items = list(set(df['item_id']))

len(all_users), len(all_items), df.shape[0]

(6040, 3706, 1000209)

In [6]:
movie_map = {p:i for i,p in enumerate(all_items)}
user_map = {u:i for i,u in enumerate(all_users)}

len(user_map.keys()), len(movie_map.keys()), df.shape[0]

(6040, 3706, 1000209)

ncf에서 user, item, interaction 수와 정확히 일치하며, README를 통해 20개 이상의 평가를 수행한 유저만 있는 것을 알 수 있음. 위처럼 평가가 있는 모든 데이터를 positive로 함.

# Prepare Train/Test data

- leave-one-out evaluation (positive)
    - 테스트 데이터: 유저의 latest 데이터
    - 학습 데이터: 유저의 나머지 데이터
- 학습 시에는 모든 negative 데이터를 포함
- 평가 시에는 positive test data 하나와 나머지 샘플링한 unobserved 100개 데이터를 합하여 랭킹
- 위 랭킹 결과의 top k에 대해서 positive 가 포함되어 있으면 hit, 그 위치가 어디인지를 NDCG가 측정

In [7]:
# latest 데이터를 test 데이터로 하기 위해 내림차순으로 정렬
df.sort_values(by=['user_id', 'timestamp'], ascending=[True, False], inplace=True)
df.head()

,user_id,item_id,rating,timestamp
25,1,48,5,978824351
32,1,1566,4,978824330
34,1,1907,4,978824330
4,1,2355,5,978824291
30,1,2294,4,978824291


In [8]:
data_pos = [[id_] for id_ in all_users] # user, item, score

for i, row in df.iloc[::-1].iterrows():
    data_pos[user_map[row['user_id']]].append(movie_map[row['item_id']])

In [9]:
train_data = []
val_data = []
test_data = []
avg_pos = 0

for history in tqdm(data_pos):
    avg_pos += len(history)-1
    test_data.append([history[-1]])
    val_data.append([history[-2]])
    train_data.append(history[-203:-3])

print('average history length: ', avg_pos/len(all_users))

100%|██████████| 6040/6040 [00:00<00:00, 346579.79it/s]

average history length:  165.5975165562914


In [10]:
num_negative = 100

In [ ]:
def sampling(num_sample, all_items, without):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.choice(all_items)
        if movie_map[randint] in without:
            continue
        else:
            # sampled.append([user_map[user_board], pin_map[all_pins[randint]], 0])
            sampled.append(movie_map[randint])
            idx += 1
    return sampled

train_neg, test_neg = [], []

for i, pos in enumerate(train_data):
    train_neg.append(sampling(num_negative, all_items, pos))
    test_neg.append(sampling(100, all_items, pos))

* pinterest -> user: item1, item2, ...
* movielens -> (user, item), (user, item) (user, item)

거꾸로 돌면서 첫번째 등장한 데이터를 테스트 나머지를 트레인으로 두기
양의 데이터는 [user_id, item_id]

In [ ]:
train = []

for i, (tp, tn) in enumerate(zip(train_data, train_neg)):
    interactions = len(tp)*[1] + len(tn)*[0]
    train.extend([i, cid, interaction] for cid, interaction in zip(tp+tn, interactions))

In [ ]:
test = []

for i, (tp, tn) in enumerate(zip(test_data, test_neg)):
    interactions = [1] + len(tn)*[0]
    test.extend([i, cid, interaction] for cid, interaction in zip(tp+tn, interactions))

In [ ]:
train = np.array(train)
test = np.array(test)

In [ ]:
len(train)

In [ ]:
np.random.shuffle(train)
np.random.shuffle(test)